# DINO Notebook with U-Net refinement

A version implementing a U-Net as an addition to the DINO framework for refining the segmentation created using the following pipeline:

- A pretrained DINOv2 backbone is fed with imaris images to extract feature embeddings for all image patches
- Positive and negative points corresponding to the object that will be segmented respectively the background are sampled for each image
- The patches that each reference point belong to are then used to create a positive nad a negative protype embedding which are used for computing a cosine similarity between the image and prototype patches
- The cosine similarity is used for constructing a consine similarity heatmap, which is used to create a segmentation mask by applying a threshold to include only the most similar patches
- A U-Net is fed with either the raw image, the DINO heatmap, the raw image and the DINO heatmap or the raw image, the heatmap and the DINO segmentation with the aim of refining the segmentation. The U-Net loss is computed against ground truth masks created by an expert in the field, implying that this part of the pipeline introduces a supervised step. It should thus be viewed as an addition rather than a part of the pipeline

The overall goal is to invesigate whether this self supervised approach using only DINO and the supervised addition can help improve the segmentation quality. 

# 1. Setup

In [1]:
!pip -q install SciencePlots
!pip -q install imaris-ims-file-reader

import copy
import gc
from collections import Counter
import random
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.transforms import functional as TF
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import scipy.ndimage as ndi
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import scienceplots

from imaris_ims_file_reader.ims import ims
import seaborn as sns
from sklearn.metrics import average_precision_score, roc_auc_score
from skimage.transform import resize
import umap

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# The ImageNet mean and standard deviation that Meta uses
imagenet_mn = torch.tensor([0.485, 0.456, 0.406], device=device).view(1,3,1,1)
imagenet_std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1,3,1,1)

def seeding(seed):
    """Ensuring that the runs are random but reproducible"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

plt.style.use(["science", "grid", "no-latex"])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 6.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 79.5 MB/s eta 0:00:00:00:010:01


# 2. Data prepartion, prototype and evaluation functions

In [2]:
@dataclass
class Config:
    seed: int = 9
    # The input size for DINO evaluation
    # Could try using a bit larger, but performace peaks before 1400
    # Has to be a multiple of 14 as that is the patch size for v2
    vit_eval_size: int = 980

def prepare_cases(cases, align):
    """Used to put together a complete case with gt-mask, image and points"""
    prepared = []
    for case_idx, (img_t, gt, pos_pts, neg_pts) in enumerate(cases):
        gt = np.asarray(gt, bool)
        img_aligned, pos_aligned = align(img_t, gt.astype(np.uint8), pos_pts)
        _, neg_aligned = align(img_t, gt.astype(np.uint8), neg_pts)
        prepared.append({"case_idx": case_idx, "img": img_aligned[0].cpu().numpy(), "gt": gt, "pos_pts": pos_aligned, "neg_pts": neg_aligned})
    return prepared

def normalize_img(img, low, high):
    """Normalizing the loaded file images to exclude outliers"""
    img = np.asarray(img, np.float32)
    lo = np.percentile(img, low)
    hi = np.percentile(img, high)
    img = np.clip(img, lo, hi)
    img = (img-lo)/(hi-lo)
    return img.astype(np.float32)
    
def build_cases(file, z_list, img_channel):
    """Creating the cases with the images and gt masks for the selected z-slices"""
    cases = []
    # The Liu files have the gt mask as the last channel
    gt_channel = file.shape[1]-1
    for z in z_list:
        img = np.asarray(file[0, img_channel, z], np.float32)
        img = normalize_img(img,5,95)
        gt = np.asarray(file[0, gt_channel,z])>0
        img_t = torch.from_numpy(img)[None, ...]
        cases.append((img_t, gt, [],[]))
    return cases
    
def align_mask(img_t, gt, points):
    """Needed to avoid scale mismatches between gt-masks and images"""
    img = img_t.detach().cpu()
    _, h,w = img.shape
    hmask,wmask = gt.shape
    if (h,w)!= (hmask,wmask):
        img_aligned = TF.resize(img,[hmask,wmask])
        sy = hmask/h
        sx = wmask/w
    else:
        img_aligned = img
        sy = sx = 1
    points_aligned = [(int(y*sy), int(x*sx)) for (y,x) in points]
    return img_aligned, points_aligned

def control_img(img):
    """Making sure that the image has the shape [C,H,W]"""
    if isinstance(img, np.ndarray):
        img= torch.from_numpy(img)
    img = img.float()
    if img.ndim == 2:
        img = img.unsqueeze(0)
    return img.contiguous()

def pick_slices(z_min, z_max, n_slices):
    """Used to pick a certain number of evenly spread z-slices within a range from a file"""
    return np.linspace(z_min, z_max, n_slices).round().astype(int).tolist()

## 2.2 Evaluation metrics, thresholding and point sampling

In [3]:
def ap_roc(heat, gt):
    """Returns the ap and roc scores"""
    y_true = gt.astype(np.uint8).ravel()
    y_score = heat.astype(np.float32).ravel()
    return average_precision_score(y_true,y_score), roc_auc_score(y_true,y_score)

def dice(pred, gt):
    """Computes the dice score"""
    tp = np.logical_and(pred, gt).sum()
    fp = np.logical_and(pred, ~gt).sum()
    fn = np.logical_and(~pred, gt).sum()
    return float(2*tp/(2*tp+fp+fn))

def iou(pred, gt):
    """Gives the intersection over union, ie Jaccard index, between the predicted and gt masks"""
    tp = np.logical_and(pred, gt).sum()
    fp = np.logical_and(pred, ~gt).sum()
    fn = np.logical_and(~pred, gt).sum()
    return tp/(tp+fp+fn)

def hd95(pred, gt):
    """Hausdorff distance using the 95th percentile to measure how well the gt and prediction masks align"""
    d_pred = ndi.distance_transform_edt(~pred)
    d_gt = ndi.distance_transform_edt(~gt)
    bound_pred = pred^ndi.binary_erosion(pred)
    bound_gt = gt^ndi.binary_erosion(gt)
    d_pred_to_gt = d_gt[bound_pred]
    d_gt_to_pred = d_pred[bound_gt]
    all_distances = np.concatenate([d_pred_to_gt, d_gt_to_pred])
    return np.percentile(all_distances,95)

def thr_top_percent(heat, top):
    """Returns the top percent of the pixels with highest similarity scores to that the thresholding can 
    be done to create a binary mask"""
    thr = np.percentile(heat, 100-top)
    return (heat>=thr), thr

def sample_pos_points(gt, k_points, seed):
    """Sampling k points inside the gt-mask to mark what's interesting object"""
    rng = np.random.default_rng(seed)
    # Avoiding to sample the points too close to the boundaries
    mask = ndi.binary_erosion(gt, None, 20)
    y, x = np.where(mask)
    idx = rng.choice(len(y), k_points, True)
    return [(int(y[i]), int(x[i])) for i in idx]

def sample_neg_points(gt, k_points, seed):
    """Sampling k points outside the gt-mask to mark what's not interesting object"""
    rng = np.random.default_rng(seed)
    # Avoiding to sample the points too close to the boundaries
    # 10 is fine, can be increased to 20
    mask = ~ndi.binary_dilation(gt, None, 20)
    y, x = np.where(mask)
    idx = rng.choice(len(y), k_points, True)
    return [(int(y[i]), int(x[i])) for i in idx]

## 2.3 Feature extraction using the loaded DINO model

In [4]:
@torch.no_grad()
def get_patch_embeddings(model, img_t, img_size):
    """Grabbing the DINO patch embeddings from the pretrained backbone"""    
    patch = int(model.patch_size)
    # Repeating the first channel since RGB is expected
    x = img_t.unsqueeze(0).repeat(1,3,1,1).to(device)
    h, w = x.shape[-2:]
    # Resizing the images to our ViT size so that all patches will be comparable
    x = F.interpolate(x, (img_size, img_size), None, "bilinear")
    x = (x-imagenet_mn)/imagenet_std
    # Tested using other layers as well, but the last one performs the best
    tokens = model.get_intermediate_layers(x, n=1, reshape=False, return_class_token=False, norm=True)[0]
    h_grid = img_size//patch
    w_grid = img_size//patch
    patch_embeddings = tokens.reshape(h_grid, w_grid, -1)
    patch_embeddings = F.normalize(patch_embeddings, dim=-1)
    return patch_embeddings, h, w

@torch.no_grad()
def compute_sim_map(model, img_t, pos_proto, neg_proto, img_size, const):
    patch_embeddings, h, w = get_patch_embeddings(model, img_t, img_size)
    pos_sim = torch.einsum("hwd,kd->khw", patch_embeddings, pos_proto)
    neg_sim = torch.einsum("hwd,kd->khw", patch_embeddings, neg_proto)
    # Using mean aggregation here, proved to work the best
    pos_map = pos_sim.mean(dim=0)
    neg_map = neg_sim.mean(dim=0)
    # Combined similarity score
    sim_grid = pos_map - const*neg_map
    sim_upsampled = F.interpolate(sim_grid[None, None], (h, w), None, "bilinear", False)[0, 0]
    return sim_upsampled.detach().cpu().numpy().astype(np.float32)

def get_heatmap(model, img, pos_pts, neg_pts, cfg, const):
    """Generate a DINO similarity heatmap from positive and negative reference points"""
    pos_proto = proto_from_ref(model, img, pos_pts, cfg.vit_eval_size)
    neg_proto = proto_from_ref(model, img, neg_pts, cfg.vit_eval_size)
    heatmap = compute_sim_map(model, img, pos_proto, neg_proto, cfg.vit_eval_size, const)
    return heatmap

@torch.no_grad()
def proto_from_ref(model, img_t, ref_points, img_size):
    """Gets the protoype embeddings using the provided ref points"""
    patch_embeddings, h, w = get_patch_embeddings(model, img_t, img_size)
    h_grid, w_grid = patch_embeddings.shape[:2]
    protos = []
    # Making sure that the points align with the coords from the image
    for y, x in ref_points:
        y_patch = int(np.clip((y/h) * h_grid, 0, h_grid-1))
        x_patch = int(np.clip((x/w) * w_grid, 0, w_grid-1))
        protos.append(patch_embeddings[y_patch, x_patch])
    return torch.stack(protos, dim=0)

@torch.no_grad()
def proto_from_grid(patch_embeddings, h, w, ref_points):
    """Grabbing the prototype embeddings from a precomputed grid, which makes this approach much faster than proto_from_ref"""
    h_grid, w_grid = patch_embeddings.shape[:2]
    ref_points = np.asarray(ref_points)
    ys = ref_points[:,0]
    xs = ref_points[:,1]
    y_patch = np.clip((ys/h * h_grid).astype(int), 0, h_grid-1)
    x_patch = np.clip((xs/w * w_grid).astype(int), 0, w_grid-1)
    y_patch = torch.as_tensor(y_patch, device=patch_embeddings.device, dtype=torch.long)
    x_patch = torch.as_tensor(x_patch, device=patch_embeddings.device, dtype=torch.long)
    return patch_embeddings[y_patch, x_patch]

@torch.no_grad()
def heatmap_from_grid(patch_embeddings, h, w, pos_proto, neg_proto, const):
    """Computing the heatmap given a precomputed grid and prototype embeddings"""
    H, W, D = patch_embeddings.shape
    pos_proto = F.normalize(pos_proto, dim=-1)
    neg_proto = F.normalize(neg_proto, dim=-1)
    feats = patch_embeddings.reshape(-1, D)
    pos_map = feats@pos_proto.T
    neg_map = feats@neg_proto.T
    sim_grid = (pos_map.mean(dim=1) - const*neg_map.mean(dim=1)).reshape(H, W)
    sim_upsampled = F.interpolate(sim_grid[None, None], size=(h, w), mode="bilinear", align_corners=False)[0,0]
    return sim_upsampled.detach().cpu().numpy().astype(np.float32)
    
def evaluate_dino_backbone(model_name, model, cases, cfg, const, n_points, seeds, top_percent):
    """Evaluation of the backbone for a setup of slices"""
    rows = []
    for i, case in enumerate(cases):
        img = control_img(case["img"])
        gt = case["gt"].astype(bool)
        patch_embeddings, h, w = get_patch_embeddings(model, img, cfg.vit_eval_size)
        for seed in seeds:
            pos_pts = sample_pos_points(gt, n_points, seed)
            neg_pts = sample_neg_points(gt, n_points, seed)
            pos_proto = proto_from_grid(patch_embeddings, h, w, pos_pts)
            neg_proto = proto_from_grid(patch_embeddings, h, w, neg_pts)
            heat = heatmap_from_grid(patch_embeddings, h, w, pos_proto, neg_proto, const)
            pred, thr = thr_top_percent(heat, top_percent)
            rows.append({"model": model_name, "file": case["file"], "z": case["z"], "case": i, "seed": seed, "Dice": dice(pred, gt), "IoU": iou(pred, gt), "threshold": thr, "HD95": hd95(pred,gt)})
    return pd.DataFrame(rows)

# 3. Experiments

## 3.1 Model loading and dataset creation

In [ ]:
# Loading the models
# Could try using DINOv3 as well, need to rebuild the framework a bit but shouldn't take long
# Giant seems to perform the best - should be used in the final runs
dino2_vits14 = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14").to(device).eval()
dino2_vitb14 = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14").to(device).eval()
dino2_vitl14 = torch.hub.load("facebookresearch/dinov2", "dinov2_vitl14").to(device).eval()
dino2_vitg14 = torch.hub.load("facebookresearch/dinov2", "dinov2_vitg14").to(device).eval()

models_eval = {"dinov2_s14_pretrained": dino2_vits14, "dinov2_b14_pretrained": dino2_vitb14, "dinov2_l14_pretrained": dino2_vitl14, "dinov2_g14_pretrained": dino2_vitg14}

# Config, seeding and file loading
CFG = Config()
seeding(CFG.seed)

# The files, B lacks a lot of annotated slices and is therefore left out
# This needs to be adjusted, LiuData is provided with the notebook - can be found in input datasets
fileA = ims('/kaggle/input/datasets/matildahellstrom/liudata/10-26-40_6_Blaze_crop2 quantified.ims')
fileC = ims('/kaggle/input/datasets/matildahellstrom/liudata/16-22-39_IC 4_Trap_SOST_col1-_Blaze crop1 segmentation quantified.ims')
fileD = ims('/kaggle/input/datasets/matildahellstrom/liudata/15-15-06_IC 2_Trap_SOST_col1_Blaze crop2 segmentation quantified.ims')
fileE = ims('/kaggle/input/datasets/matildahellstrom/liudata/11-26-23_1TRAP_SOST_NCAM_Blaze crop3 segmentation quantified.ims')
fileF = ims('/kaggle/input/datasets/matildahellstrom/liudata/11-30-50_5TRAP_SOST_NCAM_Blazecrop1 segmentation quantified.ims')

print("file A shape:", fileA.shape)
print("file C shape:", fileC.shape)
print("file D shape:", fileD.shape)
print("file E shape:", fileE.shape)
print("file F shape:", fileF.shape)

# Grabbing some evenly spaced slices and avoiding beginning and end of files
z_A = pick_slices(40, 213, 100)
z_C = pick_slices(40, 592, 100)
z_D = pick_slices(40, 438, 100)
z_E = pick_slices(40, 447, 100)
z_F = pick_slices(40, 251, 100)

# Channel should be used for the segmentation, except for A - 3 is the right channel here
files_dict = {"A": {"file_obj": fileA, "z_list": z_A, "img_channel": 3},
              "C": {"file_obj": fileC, "z_list": z_C, "img_channel": 4},
              "D": {"file_obj": fileD, "z_list": z_D, "img_channel": 4},
              "E": {"file_obj": fileE, "z_list": z_E, "img_channel": 4},
              "F": {"file_obj": fileF, "z_list": z_F, "img_channel": 4}}

cases_per_file = {}
for file_name, info in files_dict.items():
    print(f"Processing file {file_name}")
    cases = build_cases(info["file_obj"], info["z_list"], info["img_channel"])
    prepared = prepare_cases(cases, align_mask)
    for i, case in enumerate(prepared):
        case["file"] = file_name
        case["z"] = info["z_list"][i]
    cases_per_file[file_name] = prepared
    print(f"n({file_name}):", len(prepared))

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 278MB/s]
Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitb14_pretrain.pth


...

100%|██████████| 4.23G/4.23G [00:15<00:00, 301MB/s] 


Opening readonly file: /kaggle/input/datasets/matildahellstrom/liudata/10-26-40_6_Blaze_crop2 quantified.ims 

Opening readonly file: /kaggle/input/datasets/matildahellstrom/liudata/16-22-39_IC 4_Trap_SOST_col1-_Blaze crop1 segmentation quantified.ims 

Opening readonly file: /kaggle/input/datasets/matildahellstrom/liudata/15-15-06_IC 2_Trap_SOST_col1_Blaze crop2 segmentation quantified.ims 

Opening readonly file: /kaggle/input/datasets/matildahellstrom/liudata/11-26-23_1TRAP_SOST_NCAM_Blaze crop3 segmentation quantified.ims 

Opening readonly file: /kaggle/input/datasets/matildahellstrom/liudata/11-30-50_5TRAP_SOST_NCAM_Blazecrop1 segmentation quantified.ims 

file A shape: (1, 5, 253, 1026, 775)
file C shape: (1, 6, 632, 1725, 2048)
file D shape: (1, 6, 478, 2933, 3050)
file E shape: (1, 6, 487, 847, 1228)
file F shape: (1, 6, 291, 1632, 1825)
Processing file A
n(A): 100
Processing file C


Exception ignored in: <function WeakValueDictionary.__init__.<locals>.remove at 0x7897c9568400>
Traceback (most recent call last):
  File "/usr/lib/python3.12/weakref.py", line 105, in remove
    def remove(wr, selfref=ref(self), _atomic_removal=_remove_dead_weakref):

KeyboardInterrupt: 
Exception ignored in: <function WeakValueDictionary.__init__.<locals>.remove at 0x7897c9568400>
Traceback (most recent call last):
  File "/usr/lib/python3.12/weakref.py", line 105, in remove
    def remove(wr, selfref=ref(self), _atomic_removal=_remove_dead_weakref):

KeyboardInterrupt: 
Exception ignored in: <function WeakValueDictionary.__init__.<locals>.remove at 0x7897c9568400>
Traceback (most recent call last):
  File "/usr/lib/python3.12/weakref.py", line 105, in remove
    def remove(wr, selfref=ref(self), _atomic_removal=_remove_dead_weakref):

KeyboardInterrupt: 


n(C): 100
Processing file D


Exception ignored in: <function WeakValueDictionary.__init__.<locals>.remove at 0x7897c9568400>
Traceback (most recent call last):
  File "/usr/lib/python3.12/weakref.py", line 105, in remove
    def remove(wr, selfref=ref(self), _atomic_removal=_remove_dead_weakref):

KeyboardInterrupt: 


## 3.2 DINO backbone experiments

In [ ]:
eval_cases = []
for f in ["A", "C", "D", "E", "F"]:
    # Using 100 slices due to computational limitations
    eval_cases.extend(cases_per_file[f][:100])
    
backbone_results = []
for model_name, model in models_eval.items():
    print("Evaluating:", model_name)
    # Setting lambda as 0.8, 25 reference points, three seeds and using 10 as the top percentage
    df_model = evaluate_dino_backbone(model_name, model, eval_cases, CFG, 0.8, 25, (0,1,2), 10)
    backbone_results.append(df_model)

df_dino_backbone = pd.concat(backbone_results, ignore_index=True)
summary_dino_backbone = (df_dino_backbone.groupby("model").agg(n_samples=("Dice", "size"), Mean_Dice=("Dice", "mean"), Std_Dice=("Dice", "std"), Mean_IoU=("IoU", "mean"), Std_IoU=("IoU", "std"), Mean_HD95=("HD95", "mean"), Std_HD95=("HD95", "std"),).reset_index())
display(summary_dino_backbone.round(4))

## 3.3 Robustness and generalization

The transfer experiments, using prototype embeddings from a atarget slice to evaluate the segmentation performance on the same slice, slices from the same file and slices from differnet files.

In [ ]:
def case_key(case):
    """Making it possible to access slices by sample and z-pos to enable resuse off precomputed embeddings"""
    return (case["file"], case["z"])

def transfer_type(ref_case, target_case):
    """Classifies the type of the transfer"""
    if ref_case["file"] == target_case["file"] and ref_case["z"] == target_case["z"]:
        return "same_slice"
    elif ref_case["file"] == target_case["file"]:
        return "same_file"
    else:
        return "other_file"

@torch.no_grad()
def precompute_embeddings(model, cases, cfg):
    """Precomputing the embeddings for the transfer experiment"""
    cache = {}
    for idx, case in enumerate(cases):
        print(f"Precomputing {idx+1}/{len(cases)}: file={case['file']}, z={case['z']}")
        img = control_img(case["img"])
        gt = case["gt"].astype(bool)
        patch_embeddings, h, w = get_patch_embeddings(model, img, cfg.vit_eval_size)
        # Using the key to store the data for the case
        cache[case_key(case)] = {"patch_embeddings": patch_embeddings, "h": h, "w": w, "gt": gt}
    return cache

@torch.no_grad()
def proto_from_cached_grid(cache_entry, ref_points):
    """Getting the prototype embeddings from the cached patch embedding grid"""
    patch_embeddings = cache_entry["patch_embeddings"]
    h = cache_entry["h"]
    w = cache_entry["w"]
    H, W,D = patch_embeddings.shape
    ref_points = np.asarray(ref_points)
    ys = ref_points[:,0]
    xs = ref_points[:,1]
    y_patch = np.clip((ys/h*H).astype(int), 0, H-1)
    x_patch = np.clip((xs/w*W).astype(int), 0, W-1)
    y_patch = torch.as_tensor(y_patch, device=patch_embeddings.device, dtype=torch.long)
    x_patch = torch.as_tensor(x_patch, device=patch_embeddings.device, dtype=torch.long)
    return patch_embeddings[y_patch, x_patch]

@torch.no_grad()
def get_ref_cached(ref_case, embedding_cache, n_points, seed):
    """Generating pos and neg prototype embeddings from a reference slice"""
    gt = ref_case["gt"].astype(bool)
    pos_pts = sample_pos_points(gt, n_points, seed)
    neg_pts = sample_neg_points(gt, n_points, seed)
    cache_entry = embedding_cache[case_key(ref_case)]
    pos_proto = proto_from_cached_grid(cache_entry, pos_pts)
    neg_proto = proto_from_cached_grid(cache_entry, neg_pts)
    return pos_proto, neg_proto, pos_pts, neg_pts

@torch.no_grad()
def segment_with_ref(target_case, pos_proto,neg_proto, embedding_cache, const, top_percent):
    """Using embeddings from a reference slice to segment a target slice"""
    cache_entry = embedding_cache[case_key(target_case)]
    patch_embeddings = cache_entry["patch_embeddings"]
    gt = cache_entry["gt"]
    H, W, D = patch_embeddings.shape
    feats = patch_embeddings.reshape(-1, D)
    pos_proto = F.normalize(pos_proto, dim=-1)
    neg_proto = F.normalize(neg_proto, dim=-1)
    pos_map = feats @ pos_proto.T
    neg_map = feats @ neg_proto.T
    sim = pos_map.mean(dim=1) - const*neg_map.mean(dim=1)
    sim_grid = sim.reshape(H, W)
    heat = F.interpolate(sim_grid[None, None], size=gt.shape, mode="bilinear", align_corners=False)[0, 0]
    heat = heat.detach().cpu().numpy().astype(np.float32)
    pred, thr = thr_top_percent(heat, top_percent)
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    return {"dice": dice(pred, gt), "iou": iou(pred, gt), "hd95": hd95(pred, gt), "thr": thr,}

def eval_transfers(model, cases_per_file, cfg, files, n_ref_per_file, n_target_per_file, n_points, seeds, const, top_percent):
    """Evaluating the prototype transfer between samples and slices"""
    rows = []
    ref_cases = []
    target_cases = []
    for f in files:
        ref_cases.extend(cases_per_file[f][:n_ref_per_file])
        target_cases.extend(cases_per_file[f][:n_target_per_file])
    all_cases_dict = {case_key(case): case for case in ref_cases + target_cases}
    all_cases = list(all_cases_dict.values())
    embedding_cache = precompute_embeddings(model, all_cases, cfg)
    for seed in seeds:
        print(f"Seed: {seed}")
        for ref_idx, ref_case in enumerate(ref_cases):
            print(f"Ref {ref_idx+1}/{len(ref_cases)}: " f"file={ref_case['file']}, z={ref_case['z']}")
            pos_proto, neg_proto, pos_pts, neg_pts = get_ref_cached(ref_case, embedding_cache, n_points, seed)
            for target_case in target_cases:
                case_type = transfer_type(ref_case, target_case)
                result = segment_with_ref(target_case, pos_proto, neg_proto, embedding_cache, const, top_percent)
                rows.append({"seed": seed, "ref_file": ref_case["file"], "ref_z": ref_case["z"], "target_file": target_case["file"], "target_z": target_case["z"], "case_type": case_type, "Dice": result["dice"], "IoU": result["iou"], "HD95": result["hd95"], "threshold": result["thr"],})
    return pd.DataFrame(rows)

df_ref_transfer = eval_transfers(models_eval["dinov2_g14_pretrained"], cases_per_file, CFG, ("A","C","D","E","F"), 5, 20, 25, (0, 1, 2), 0.8, 10)
file_order = ["A", "C", "D", "E", "F"]

pairwise_transfer_table = (df_ref_transfer.groupby(["ref_file", "target_file"], observed=True).agg(Mean_Dice=("Dice", "mean"), Std_Dice=("Dice", "std"), Mean_IoU=("IoU", "mean"), Std_IoU=("IoU", "std"), N=("Dice", "count")).reset_index())
pairwise_transfer_table["ref_file"] = pd.Categorical(pairwise_transfer_table["ref_file"], categories=file_order, ordered=True)
pairwise_transfer_table["target_file"] = pd.Categorical(pairwise_transfer_table["target_file"], categories=file_order, ordered=True)
pairwise_transfer_table = (pairwise_transfer_table.sort_values(["ref_file", "target_file"]).reset_index(drop=True))
display(pairwise_transfer_table.round(5))

summary_transfer = (df_ref_transfer.groupby("case_type", observed=True).agg(n=("Dice", "count"), Mean_Dice=("Dice", "mean"), Std_Dice=("Dice", "std"), Mean_IoU=("IoU", "mean"), Std_IoU=("IoU", "std")).reset_index())
summary_transfer["case_type"] = summary_transfer["case_type"].replace({"same_slice": "Same slice", "same_file": "Same sample", "other_file": "Different sample"})
transfer_order = ["Same slice", "Same sample", "Different sample"]
summary_transfer["case_type"] = pd.Categorical(summary_transfer["case_type"], categories=transfer_order, ordered=True)
summary_transfer = (summary_transfer.sort_values("case_type").reset_index(drop=True))
display(summary_transfer.round(5))

## 3.4 Point sensitivity

In [ ]:
@torch.no_grad()
def eval_point_sens(model, cases, cfg, n_points_list, seeds, const, top_percent):
    """Evaluating how the segmentation performance is affected by the number of chosen reference points"""
    rows = []
    max_points = max(n_points_list)
    for i, case in enumerate(cases):
        print(f"Case {i+1}/{len(cases)}: file={case['file']}, z={case['z']}")
        img = control_img(case["img"])
        gt = case["gt"].astype(bool)
        patch_embeddings, h, w = get_patch_embeddings(model, img, cfg.vit_eval_size)
        for seed in seeds:
            pos_pts_all = sample_pos_points(gt, max_points, seed)
            neg_pts_all = sample_neg_points(gt, max_points, seed)
            for n_points in n_points_list:
                pos_pts = pos_pts_all[:n_points]
                neg_pts = neg_pts_all[:n_points]
                pos_proto = proto_from_grid(patch_embeddings, h, w, pos_pts)
                neg_proto = proto_from_grid(patch_embeddings, h, w, neg_pts)
                heat = heatmap_from_grid(patch_embeddings, h, w, pos_proto, neg_proto, const)
                pred, thr = thr_top_percent(heat, top_percent)
                rows.append({"file": case["file"], "z": case["z"], "case": i, "seed": seed, "n_points": n_points, "Dice": dice(pred, gt), "IoU": iou(pred, gt), "threshold": thr})
    return pd.DataFrame(rows)

eval_cases_points = []
for f in ["A", "C", "D", "E", "F"]:
    eval_cases_points.extend(cases_per_file[f][:25])
df_point_sensitivity = eval_point_sens(dino2_vitg14, eval_cases_points, CFG, (1, 3, 5, 10, 15, 25, 50), (0, 1, 2, 3, 4), 0.8, 10)
display(df_point_sensitivity.round(4))

## 3.5 Threshold sensitivity

In [ ]:
@torch.no_grad()
def evaluate_threshold_sensitivity_fast(model, cases, cfg, const, n_points, seed, top_percents, compute_hd95):
    """Evaluation of how the chosen top percentage value affects the obtained segmentation performance"""
    rows = []
    top_percents = sorted(top_percents)
    for i, case in enumerate(cases):
        print(f"Case {i+1}/{len(cases)}: file={case.get('file')}, z={case.get('z')}")
        img = control_img(case["img"])
        gt = case["gt"].astype(bool)
        pos_pts = sample_pos_points(gt, n_points, seed)
        neg_pts = sample_neg_points(gt, n_points, seed)
        heat = get_heatmap(model, img, pos_pts, neg_pts, cfg,const)
        flat = heat.ravel()
        for tp in top_percents:
            thr = np.percentile(flat, 100-tp)
            pred = heat >= thr
            row = {"case": i, "file": case.get("file"), "z": case.get("z"), "top_percent": tp, "Dice": dice(pred, gt), "IoU": iou(pred, gt), "threshold": thr,}
            if compute_hd95:
                row["HD95"] = hd95(pred, gt)
            rows.append(row)
    return pd.DataFrame(rows)
    
eval_cases_thresh = []
for f in ["A", "C", "D", "E", "F"]:
    eval_cases_thresh.extend(cases_per_file[f][:25])
df_threshold = evaluate_threshold_sensitivity_fast(dino2_vitg14, eval_cases_thresh, CFG, 0.8, 25, 0, (2,4,6,8,10,12,14,16,18,20), False)
summary_threshold = (df_threshold.groupby("top_percent").agg(Mean_Dice=("Dice", "mean"), Std_Dice=("Dice", "std"), N=("Dice", "count")).reset_index())
summary_threshold["SEM_Dice"] = (summary_threshold["Std_Dice"] / np.sqrt(summary_threshold["N"]))
summary_threshold["CI95_Dice"] = (1.96*summary_threshold["SEM_Dice"])

display(summary_threshold.sort_values("Mean_Dice", ascending=False)[["top_percent", "Mean_Dice", "Std_Dice", "CI95_Dice"]].round(4))

# 4. U-net addition

Training a U-Net on either 1) the raw image, 2) the raw image and the DINO heatmap or 3) the raw image, the DINO heatmap and the DINO segmentation mask.

In [ ]:
class UnetPrecomp(Dataset):
    """Combining the GT-mask and the corresponding precomputed input"""
    def __init__(self, precomputed_items):
        self.items = precomputed_items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        x = self.items[idx]["x"].float()
        y = self.items[idx]["y"].float()
        return x, y

class DoubleConv(nn.Module):
    """Convoltuion block used in the encoder and decoder"""
    def __init__(self, in_c, out_c):
        super().__init__()
        self.double_conv = nn.Sequential(nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.double_conv(x)

class DownBlock(nn.Module):
    """Encoder block: downsamples and returns skip connection features"""
    def __init__(self, in_c, out_c):
        super().__init__()
        self.double_conv = DoubleConv(in_c,out_c)
        self.down_sample = nn.MaxPool2d(2)

    def forward(self, x):
        skip_out = self.double_conv(x)
        down_out = self.down_sample(skip_out)
        return down_out, skip_out

class UpBlock(nn.Module):
    """Decoder block: upsampling and using the skip connection features"""
    def __init__(self, in_c, out_c):
        super().__init__()
        self.up_sample = nn.ConvTranspose2d(in_c-out_c, in_c-out_c, 2, 2)
        self.double_conv = DoubleConv(in_c, out_c)

    def forward(self, down_input, skip_input):
        x = self.up_sample(down_input)
        x = torch.cat([x, skip_input], dim=1)
        return self.double_conv(x)

class UNet(nn.Module):
    """Creates the full U-net model, outputs logits"""
    def __init__(self, in_c=2, out_c=1):
        super().__init__()
        self.down_conv1 = DownBlock(in_c, 32)
        self.down_conv2 = DownBlock(32, 64)
        self.double_conv = DoubleConv(64, 128)
        self.up_conv2 = UpBlock(192, 64)
        self.up_conv1 = UpBlock(96, 32)
        self.conv_last = nn.Conv2d(32, out_c, kernel_size=1)

    def forward(self, x):
        x, skip1 = self.down_conv1(x)
        x, skip2 = self.down_conv2(x)
        x = self.double_conv(x)
        x = self.up_conv2(x, skip2)
        x = self.up_conv1(x, skip1)
        x = self.conv_last(x)
        return x

## 4.1 Loss and training functions

In [ ]:
def combo_loss_dice(log, targ):
    """Computes dice score inspired by the paper by Taghanaki et al."""
    probs = torch.sigmoid(log)
    dice_score = 2*(probs*targ).sum(dim=(1,2,3))/(probs.sum(dim=(1,2,3))+targ.sum(dim=(1,2,3)))
    return 1-dice_score.mean()

def combo_loss(log, targ):
    """Combines BCE and dice loss, weighting them 50/50"""
    bce = F.binary_cross_entropy_with_logits(log, targ)
    dloss = combo_loss_dice(log, targ)
    return bce/2+dloss/2

def one_epoch(loader, model,optimizer):
    """Training one epoch and returns the training loss"""
    model.train()
    losses = []
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = combo_loss(logits, y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return float(np.mean(losses))

def run_unet(train_loader, val_loader, m_class, m_kwargs, run_name, variant, lr, num_epochs, thresholds, patience):
    """Training the U-net and saving the best epoch in terms of dice score"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    model = m_class(**m_kwargs).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr)
    best_val_dice = -1
    best_epoch = -1
    best_thr = None
    history = []
    path = f"best_unet_{run_name}.pt"
    no_improve = 0

    for epoch in range(num_epochs):
        train_loss = one_epoch(train_loader, model, optimizer)
        # Using the approach where several thresholds are tested and then saving the best one
        df_sweep = thr_sweep(val_loader, model, thresholds)
        best_row = df_sweep.iloc[0]
        val_dice = float(best_row["Dice_mean"])
        val_iou = float(best_row["IoU_mean"])
        thr = float(best_row["threshold"])
        history.append({"epoch": epoch+1, "train_loss": train_loss, "val_dice": val_dice, "val_iou": val_iou, "best_thr": thr})
        # Finding the best epochs by measuring dice
        if val_dice > best_val_dice:
            best_val_dice = val_dice
            best_epoch = epoch+1
            best_thr = thr
            torch.save(model.state_dict(), path)
            no_improve = 0
        else:
            no_improve+=1
        print(f"Epoch {epoch+1:02d}: train_loss={train_loss:.4f}, val_dice={val_dice:.4f}, val_iou={val_iou:.4f}, best_thr={thr:.2f}")
        if no_improve >= patience:
            print("Early stopping triggered")
            break

    model.load_state_dict(torch.load(path, map_location=device))
    df_best_sweep = thr_sweep(val_loader, model, thresholds)
    history_df = pd.DataFrame(history)

    print("\nBest epoch:", best_epoch)
    print("Best val Dice:", round(best_val_dice,4))
    print("Best threshold:", best_thr)
    print("Checkpoint saved to:", path)
    display(history_df)

    return {"run_name": run_name, "variant": variant, "model": model, "best_val_dice": best_val_dice, "best_val_iou": float(df_best_sweep.iloc[0]["IoU_mean"]), "best_thr": best_thr,
        "best_epoch": best_epoch, "history_df": history_df.copy(), "df_best_sweep": df_best_sweep.copy(), "path": path}

## 4.2 Input creation and data loading

In [ ]:
def build_raw_input(img):
    """Creating the raw image input"""
    raw = np.asarray(img, dtype=np.float32)
    raw = (raw-raw.min())/(raw.max()-raw.min())
    x = raw[None, ...]
    return torch.from_numpy(x)

def build_heat_input(heatmap):
    """Creating the heatmap input"""
    heat = np.asarray(heatmap, dtype=np.float32)
    heat = (heat-heat.min())/(heat.max()-heat.min())
    x = heat[None, ...]
    return torch.from_numpy(x)
    
def build_rawheat_input(img, heatmap):
    """Creating the raw image and heatmap input"""
    raw = np.asarray(img, dtype=np.float32)
    heat = np.asarray(heatmap, dtype=np.float32)
    raw = (raw-raw.min())/(raw.max()-raw.min())
    heat = (heat-heat.min())/(heat.max()-heat.min())
    x = np.stack([raw,heat], axis=0)
    return torch.from_numpy(x)

def build_rawheatmask_input(img, heatmap, pred_dino):
    """Creating the raw image, heatmap input and binary mask input"""
    raw = np.asarray(img, dtype=np.float32)
    heat = np.asarray(heatmap, dtype=np.float32)
    mask = np.asarray(pred_dino, dtype=np.float32)
    raw = (raw-raw.min())/(raw.max()-raw.min())
    heat = (heat-heat.min())/(heat.max()-heat.min())
    x = np.stack([raw, heat, mask], axis=0)
    return torch.from_numpy(x)

@torch.no_grad()
def thr_sweep(loader, model, thresholds):
    """Evaluates thresholds and returns the one with the highest dice"""
    model.eval()
    prob_list, targ_list, rows = [],[],[]
    for x, y in loader:
        x = x.to(device)
        logits = model(x)
        probs = torch.sigmoid(logits).squeeze(1).cpu().numpy()
        targets = y.squeeze(1).cpu().numpy()
        prob_list.extend(list(probs))
        targ_list.extend(list(targets))
    for thr in thresholds:
        dices = []
        ious = []
        for probs, gt in zip(prob_list, targ_list):
            pred = probs >= thr
            gt_bool = gt.astype(bool)
            dices.append(dice(pred, gt_bool))
            ious.append(iou(pred, gt_bool))
        rows.append({"threshold": thr, "Dice_mean": np.mean(dices), "Dice_std": np.std(dices), "IoU_mean": np.mean(ious), "IoU_std": np.std(ious)})
    return pd.DataFrame(rows).sort_values("Dice_mean", ascending=False).reset_index(drop=True)

def precompute_cases(cases, model_dino, cfg, variant, const, mode, n_points, seed, top_percent, targ_size):
    """Precomputes the U-Net inputs and GT-masks"""
    precomputed = []
    for case in cases:
        img_t = control_img(case["img"])
        gt = case["gt"].astype(np.float32)
        pos_pts = sample_pos_points(gt.astype(bool), n_points, seed)
        neg_pts = sample_neg_points(gt.astype(bool), n_points, seed)
        heat = get_heatmap(model_dino, img_t, pos_pts, neg_pts, cfg, const)
        raw_img = img_t[0].cpu().numpy()
        if variant == "raw":
            x = build_raw_input(raw_img).float()
        elif variant == "heat":
            x = build_heat_input(heat).float()
        elif variant == "rawheat":
            x = build_rawheat_input(raw_img, heat).float()
        elif variant == "rawheatmask":
            pred_dino, _ = thr_top_percent(heat, top_percent)
            x = build_rawheatmask_input(raw_img, heat, pred_dino).float()
        y = torch.from_numpy(gt)[None, ...].float()
        # Resizing to targ_size so that the U-Net input is tha same size
        x = F.interpolate(x[None], targ_size, None, "bilinear", False)[0]
        y = F.interpolate(y[None], targ_size, None, "nearest")[0]
        precomputed.append({"x": x, "y": y})
    return precomputed

def build_loaders(train_cases, val_cases, model_dino, cfg, variant, batch_size, const, mode, n_points, seed, top_percent, targ_size):
    """Creating the train and validation loaders"""
    train_precomputed = precompute_cases(train_cases, model_dino, cfg, variant, const, mode, n_points, seed, top_percent, targ_size)
    val_precomputed = precompute_cases(val_cases, model_dino, cfg, variant, const, mode, n_points, seed, top_percent, targ_size)
    train_dataset = UnetPrecomp(train_precomputed)
    val_dataset = UnetPrecomp(val_precomputed)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader

## 4.3 U-net and DINO prediction comparison

In [ ]:
@torch.no_grad()
def pred_raw(unet_model, raw_img, threshold, targ_size):
    """Predicting mask for raw and heat input"""
    org_h, org_w = raw_img.shape
    x = build_raw_input(raw_img).float()[None]
    # Resizing to the scale we used for training
    x = F.interpolate(x, targ_size, None, "bilinear", False).to(device)
    logits = unet_model(x)
    probs_small = torch.sigmoid(logits)
    probs = F.interpolate(probs_small, (org_h, org_w), None, "bilinear", False)[0,0].cpu().numpy()
    pred = probs >= threshold
    return probs, pred

@torch.no_grad()
def pred_heat(unet_model, heatmap, threshold, targ_size):
    """Predicts mask from heatmap-only input"""
    org_h, org_w = heatmap.shape
    x = build_heat_input(heatmap).float()[None]
    x = F.interpolate(x, targ_size, None, "bilinear", False).to(device)
    logits = unet_model(x)
    probs_small = torch.sigmoid(logits)
    probs = F.interpolate(probs_small, (org_h,org_w), None, "bilinear", False)[0,0].cpu().numpy()
    pred = probs >= threshold
    return probs, pred
    
@torch.no_grad()
def pred_rawheat(unet_model, raw_img, heatmap, threshold, targ_size):
    """Predicting mask for raw and heat input"""
    org_h, org_w = raw_img.shape
    x = build_rawheat_input(raw_img, heatmap).float()[None]
    x = F.interpolate(x, targ_size, None, "bilinear", False).to(device)
    logits = unet_model(x)
    probs_small = torch.sigmoid(logits)
    probs = F.interpolate(probs_small, (org_h,org_w), None, "bilinear", False)[0,0].cpu().numpy()
    pred = probs >= threshold
    return probs, pred
    
@torch.no_grad()
def pred_rawheatmask(unet_model, raw_img, heatmap, pred_dino, threshold, targ_size):
    """Predicting mask for raw, heat and DINO mask input"""
    org_h, org_w = raw_img.shape
    x = build_rawheatmask_input(raw_img, heatmap, pred_dino).float()[None]
    x = F.interpolate(x, targ_size, None, "bilinear", False).to(device)
    logits = unet_model(x)
    probs_small = torch.sigmoid(logits)
    probs = F.interpolate(probs_small, (org_h,org_w), None, "bilinear", False)[0,0].cpu().numpy()
    pred = probs >= threshold
    return probs, pred

@torch.no_grad()
def dino_vs_unet(unet_model, prepared_cases, model_dino, variant, threshold, const, targ_size, cfg, n_points, seed, top_percent):
    """Comparing the DINO and U-net results per case"""
    rows = []
    for i, case in enumerate(prepared_cases):
        img = control_img(case["img"])
        gt = case["gt"].astype(bool)
        
        pos_pts = sample_pos_points(gt, n_points, seed)
        neg_pts = sample_neg_points(gt, n_points, seed)
        heat = get_heatmap(model_dino,img, pos_pts, neg_pts, cfg, const)
        pred_dino, _ = thr_top_percent(heat, top_percent)
        raw_img = img[0].cpu().numpy()
        
        if variant == "raw":
            _, pred_unet = pred_raw(unet_model, raw_img, threshold, targ_size)
        elif variant == "heat":
            _, pred_unet = pred_heat(unet_model, heat, threshold, targ_size)
        elif variant == "rawheat":
            _, pred_unet = pred_rawheat(unet_model, raw_img, heat, threshold, targ_size)
        elif variant == "rawheatmask":
            _, pred_unet = pred_rawheatmask(unet_model, raw_img, heat, pred_dino, threshold, targ_size)
        rows.append({"case": i, "Dice_DINO": dice(pred_dino, gt), "Dice_UNET": dice(pred_unet, gt), "IoU_DINO": iou(pred_dino, gt),
            "IoU_UNET": iou(pred_unet, gt), "HD95_DINO": hd95(pred_dino, gt), "HD95_UNET": hd95(pred_unet, gt),})
    
    df_compare = pd.DataFrame(rows)
    return df_compare

## 4.4 Test run

In [ ]:
model_dino = models_eval["dinov2_s14_pretrained"]
files = ["A", "C", "D", "E", "F"]
seeds = [0]
variant_cfgs = {"raw": {"in_c": 1}, "heat": {"in_c": 1}, "rawheat": {"in_c": 2}, "rawheatmask": {"in_c": 3}}
all_compare = []
saved_results = {}

for file in files:
    print(f"Validation file: {file}")
    train_files = [i for i in ["A", "C", "D", "E", "F"] if i != file]
    for seed in seeds:
        print(f"Seed: {seed}")
        for variant_name, variant_info in variant_cfgs.items():
            print(f"\nVariant: {variant_name}")
            train_cases = []
            for f in train_files:
                train_cases.extend(cases_per_file[f][:25])
            val_cases = cases_per_file[file][:25]
            print("Train files:", train_files)
            print("Val file:", file)
            train_loader, val_loader = build_loaders(train_cases, val_cases, model_dino, CFG, variant_name, 1, 0.8, "mean", 25, seed, 10, (980, 980))
            results = run_unet(train_loader, val_loader, UNet, {"in_c": variant_info["in_c"], "out_c": 1}, f"unet_{variant_name}_{file}_seed_{seed}", variant_name, 1e-3, 15, (0.2, 0.4, 0.6, 0.8), 5)
            saved_results[(file, seed, variant_name)] = {"model": results["model"], "best_thr": results["best_thr"], "val_cases": val_cases}
            
            df = dino_vs_unet(results["model"], val_cases, model_dino, variant_name, results["best_thr"], 0.8, (980,980), CFG, 25, 0, 10)
            df["file"] = file
            df["seed"] = seed
            df["variant"] = variant_name
            df["best_thr"] = results["best_thr"]
            df["best_val_dice"] = results["best_val_dice"]
            df["best_epoch"] = results["best_epoch"]
            all_compare.append(df)

df_compare_models = pd.concat(all_compare, ignore_index=True)
display(df_compare_models.round(4))

In [ ]:
summary_main = (df_compare_models.groupby(["variant", "file"]).agg(n_samples=("Dice_UNET", "size"), Mean_Dice_DINO=("Dice_DINO", "mean"), Mean_Dice_UNET=("Dice_UNET", "mean"), Mean_HD95_DINO=("HD95_DINO", "mean"), Mean_HD95_UNET=("HD95_UNET", "mean")).reset_index())
variant_labels = {"raw": "Raw", "heat": "Heatmap", "rawheat": "Raw + Heatmap", "rawheatmask": "Raw + Heatmap + Mask",}
variant_order = ["raw", "heat", "rawheat", "rawheatmask"]
summary_main["variant_label"] = summary_main["variant"].map(variant_labels)
summary_main["variant"] = pd.Categorical(summary_main["variant"], categories=variant_order, ordered=True)
summary_main = summary_main.sort_values(["variant", "file"])
display(summary_main)

# 5. Plots

## 5.1 Segmentation performance for each backbone

Using the results from the DINO backbone experiments for plottting the mean dice scores and the 95% confidence interval. Note the black graph corresponding to the mean performace for all backbone sizes.

In [ ]:
df_backbone = df_dino_backbone.copy()
df_backbone["Backbone"] = df_backbone["model"].map(model_labels)
df_backbone["Backbone"] = pd.Categorical(df_backbone["Backbone"], categories=backbone_order, ordered=True)

summary_file = (df_backbone.groupby(["file","Backbone"], observed=True).agg(Mean_Dice=("Dice", "mean"), Std_Dice=("Dice", "std"), N=("Dice", "count")).reset_index())
summary_file["SEM_Dice"] = summary_file["Std_Dice"]/np.sqrt(summary_file["N"])
summary_file["CI95_Dice"] = 1.96*summary_file["SEM_Dice"]

summary_overall = (df_backbone.groupby("Backbone", observed=True).agg(Mean_Dice=("Dice", "mean"), Std_Dice=("Dice", "std"), N=("Dice", "count")).reset_index())
summary_overall["SEM_Dice"] = summary_overall["Std_Dice"]/np.sqrt(summary_overall["N"])
summary_overall["CI95_Dice"] = 1.96*summary_overall["SEM_Dice"]

x_positions = {backbone: i for i, backbone in enumerate(backbone_order)}
fig, ax = plt.subplots(figsize=(6,4))
for file_name, sub in summary_file.groupby("file"):
    sub = sub.sort_values("Backbone")
    x = [x_positions[str(b)] for b in sub["Backbone"]]
    ax.plot(x, sub["Mean_Dice"], marker="o", ms=4, label=f"Sample {file_name}")
    ax.errorbar(x, sub["Mean_Dice"], yerr=sub["CI95_Dice"], fmt="none", capsize=3, linewidth=0.8, alpha=0.75)
summary_overall = summary_overall.sort_values("Backbone")
x_overall = [x_positions[str(b)] for b in summary_overall["Backbone"]]

ax.plot(x_overall, summary_overall["Mean_Dice"], marker="o", linewidth=1.2, markersize=4, color="black", label="Overall mean")
ax.errorbar(x_overall, summary_overall["Mean_Dice"], yerr=summary_overall["CI95_Dice"], fmt="none", color="black", capsize=5, linewidth=1)
ax.set_xticks(range(len(backbone_order)))
ax.set_xticklabels(backbone_order)
ax.set_xlabel("Backbone")
ax.set_ylabel("Mean dice")
ax.set_ylim(0.55, 0.8)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False, fontsize=9, bbox_to_anchor=(1.02,1), loc="upper left")
plt.tight_layout()
plt.show()

## 5.2 Point and threshold sensitivity
Displaying the results from the point and threshold sanity check experiments and plotting the mean dice scores as a function of the number of reference points.

In [ ]:
summary_points = (df_point_sensitivity.groupby("n_points").agg(Mean_Dice=("Dice", "mean"), Std_Dice=("Dice", "std"), N=("Dice", "count")).reset_index())
summary_points["SEM_Dice"] = (summary_points["Std_Dice"]/np.sqrt(summary_points["N"]))
summary_points["CI95_Dice"] = (1.96*summary_points["SEM_Dice"])
display(summary_points.round(4))

plt.rcParams.update({"font.size": 12, "axes.labelsize": 12, "xtick.labelsize": 11, "ytick.labelsize": 11, "legend.fontsize": 11})

fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(summary_points["n_points"], summary_points["Mean_Dice"], yerr=summary_points["CI95_Dice"], marker="o", capsize=4, linewidth=1.8, label="Mean dice")
ax.axvline(25, color="black", linestyle="--", linewidth=1, label="Used setting")
ax.set_xlabel("Number of reference points")
ax.set_ylabel("Mean dice")
ax.set_ylim(0, 1)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

display(summary_threshold.round(4))

fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(summary_threshold["top_percent"],summary_threshold["Mean_Dice"], yerr=summary_threshold["CI95_Dice"], marker="o", capsize=4, linewidth=1.8, label="Mean dice")
ax.axvline(10, color="black", linestyle="--", linewidth=1, label="Used setting")
ax.set_xlabel("Top-percent threshold")
ax.set_ylabel("Mean dice")
ax.set_ylim(0, 1)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## 5.3 Transfer matrix

Visualizing the transfer matrix containing the obtained results during the transfer experiments - i.e. the dice scores for the diffrent reference target samples.

In [ ]:
transfer_matrix = (df_ref_transfer.groupby(["ref_file", "target_file"]).agg(Mean_Dice=("Dice", "mean")).reset_index())
dice_matrix = transfer_matrix.pivot(index="ref_file", columns="target_file", values="Mean_Dice")
file_order = ["A", "C", "D", "E", "F"]
dice_matrix = dice_matrix.loc[file_order, file_order]
plt.figure(figsize=(7,6))
ax = sns.heatmap(dice_matrix, annot=True, fmt=".3f", cmap="mako", vmin=0.2, vmax=0.8, square=True, linewidths=0.75, annot_kws={"fontsize": 11}, cbar_kws={"label": "Mean dice score", "shrink": 0.85})
ax.set_xlabel("Target sample", fontsize=12)
ax.set_ylabel("Reference sample", fontsize=12)
ax.tick_params(labelsize=11)
plt.tight_layout()
plt.show()

## 5.4 Qualitative results

Displaying the worst, median and best dice score cases for each file using the Giant backbone. 

In [ ]:
@torch.no_grad()
def qualitative_res(df_results, cases_per_file, model, cfg, files, const, n_points, top_percent): 
    fig, axes = plt.subplots(5, 3, figsize=(10,14))
    col_titles = ["Worst", "Median", "Best"]

    for row_idx, file in enumerate(files):
        df_file = (df_results[df_results["file"] == file].sort_values("Dice").reset_index(drop=True))
        selected_rows = [df_file.iloc[0], df_file.iloc[len(df_file)//2], df_file.iloc[-1]]

        for col_idx, row in enumerate(selected_rows):
            z = row["z"]
            seed = int(row["seed"])
            case = next(case for case in cases_per_file[file] if case["z"] == z)
            img = control_img(case["img"])
            raw = img[0].cpu().numpy()
            gt = case["gt"].astype(bool)
            pos_pts = sample_pos_points(gt, n_points, seed)
            neg_pts = sample_neg_points(gt, n_points, seed)
            heat = get_heatmap(model, img, pos_pts, neg_pts, cfg, const)
            pred, _ = thr_top_percent(heat, top_percent)
            current_dice = dice(pred, gt)
            ax = axes[row_idx, col_idx]
            ax.imshow(raw, cmap="gray")
            ax.contour(pred.astype(float), levels=[0.5], colors="dodgerblue", linewidths=1.3)
            ax.contour(gt.astype(float), levels=[0.5], colors="crimson", linewidths=1.3)
            if row_idx == 0:
                ax.set_title(col_titles[col_idx], fontsize=12)
            ax.text(0.02, 0.95, f"File {file}\nz={z}\nDice={current_dice:.3f}", transform=ax.transAxes, fontsize=9, va="top", ha="left", bbox=dict( facecolor="white", alpha=0.75, edgecolor="none"))
            ax.axis("off")

    handles = [plt.Line2D([0], [0], color="dodgerblue", lw=2, label="Prediction"), plt.Line2D([0],[0], color="crimson", lw=2, label="Ground truth")]
    fig.legend(handles=handles, loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 0.01))
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.05)
    plt.show()

df_giant = df_dino_backbone[df_dino_backbone["model"] == "dinov2_g14_pretrained"]
qualitative_res(df_giant, cases_per_file, dino2_vitg14, CFG, ("A", "C", "D", "E", "F"), 0.8, 25, 10)

## 5.5 Qualitative U-Net results 
Displaying the lowest, median and highest gain in the dice score for each file compared to the Small DINO backbone results. Due to computational limiitations, the Small backbone is the best option due to its high segmentation performance and low computational costs.

In [ ]:
@torch.no_grad()
def qualitative_unet(df_compare_models, cases_per_file, saved_results, model_dino, cfg, files, variant,const, n_points, top_percent, targ_size):
    fig, axes = plt.subplots(5, 3, figsize=(10,14))
    col_titles = ["Lowest gain","Median gain", "Highest gain"]
    
    for row_idx, file in enumerate(files):
        df_file = df_compare_models[(df_compare_models["file"] == file) & (df_compare_models["variant"] == variant)].copy()
        df_file["Dice_gain"] = df_file["Dice_UNET"] - df_file["Dice_DINO"]
        df_file = df_file.sort_values("Dice_gain").reset_index(drop=True)
        selected_rows = [df_file.iloc[0], df_file.iloc[len(df_file)//2], df_file.iloc[-1]]

        for col_idx, row in enumerate(selected_rows):
            case_idx = int(row["case"])
            seed = int(row["seed"])
            case = cases_per_file[file][case_idx]
            z = case.get("z", case_idx)
            img = control_img(case["img"])
            raw = img[0].cpu().numpy()
            gt = case["gt"].astype(bool)

            pos_pts = sample_pos_points(gt, n_points, seed)
            neg_pts = sample_neg_points(gt, n_points, seed)
            heat = get_heatmap(model_dino, img, pos_pts, neg_pts, cfg, const)
            pred_dino, _ = thr_top_percent(heat, top_percent)
            model_info = saved_results[(file, seed, variant)]
            model_unet = model_info["model"]
            thr_unet = model_info["best_thr"]

            if variant == "raw":
                _, pred_unet = pred_raw(model_unet, raw, thr_unet, targ_size)
            elif variant == "heat":
                _, pred_unet = pred_heat(model_unet, heat, thr_unet, targ_size)
            elif variant == "rawheat":
                _, pred_unet = pred_rawheat(model_unet, raw, heat, thr_unet, targ_size)
            elif variant == "rawheatmask":
                _, pred_unet = pred_rawheatmask(model_unet, raw, heat, pred_dino, thr_unet, targ_size)
            else:
                raise ValueError(f"Unknown variant: {variant}")

            dice_dino = dice(pred_dino, gt)
            dice_unet = dice(pred_unet, gt)
            dice_gain = dice_unet - dice_dino
            ax = axes[row_idx, col_idx]
            ax.imshow(raw, cmap="gray")
            ax.contour(pred_dino.astype(float), levels=[0.5], colors="dodgerblue", linewidths=1.2,linestyles="--")
            ax.contour(pred_unet.astype(float), levels=[0.5], colors="lime", linewidths=1.4)
            ax.contour(gt.astype(float), levels=[0.5], colors="crimson", linewidths=1.3)
            if row_idx == 0:
                ax.set_title(col_titles[col_idx], fontsize=12)
            ax.text(0.02, 0.96, f"File {file}, z={z}\nDINO={dice_dino:.3f}\nU-Net={dice_unet:.3f}\nΔ={dice_gain:+.3f}",
                transform=ax.transAxes, fontsize=8, va="top", ha="left", bbox=dict(facecolor="white", alpha=0.75, edgecolor="none"))
            ax.axis("off")
            
    handles = [plt.Line2D([0], [0], color="dodgerblue", lw=2, label="DINO"), plt.Line2D([0], [0], color="lime", lw=2, label="U-Net refinement"), plt.Line2D([0], [0], color="crimson", lw=2, label="Ground truth")]
    fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 0.01))
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.05)
    plt.show()

qualitative_unet(df_compare_models, cases_per_file, saved_results, dino2_vitg14, CFG, ("A", "C", "D", "E", "F"), "rawheat", 0.8, 25, 10, (980, 980))

## 5.6 U-Net gain

Visulaizing the U-Net gain when compared to the DINO Small backbone segmentation performance. The gain is computed to be the U-Net dice minus the DINO Small dice score. The 95% confidence interval is showed by the error bars.

In [ ]:
df_gain = df_compare_models.copy()
df_gain["Dice_gain"] = df_gain["Dice_UNET"] - df_gain["Dice_DINO"]
df_gain["variant_label"] = df_gain["variant"].map(variant_labels)
df_gain = df_gain[df_gain["variant"].isin(variant_order)].copy()

plt.figure(figsize=(8, 5))
ax = sns.barplot(data=df_gain, x="file", y="Dice_gain", hue="variant_label", order=["A", "C", "D", "E", "F"], hue_order=[variant_labels[v] for v in variant_order], errorbar=("ci", 95), capsize=0.08, err_kws={"linewidth": 1})
ax.axhline(0, color="black")
ax.set_xlabel("Sample")
ax.set_ylabel("Delta dice")
ax.legend(title="U-Net input", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
sns.despine()
plt.tight_layout()
plt.show()